处理所有文件的icohp数据

In [41]:
import pandas as pd
import os
import shutil

In [38]:
def parse_icohplist(filename: str) -> dict:
    data_rows = []
    current_spin = None
    spin_map = { "1" : "up", "2" : "down"}
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('COHP#'):
                if "spin" in line:
                    current_spin = line.split()[-1] # 获得自旋编号，按照icohp.lobster的习惯，1是上自旋，2是下自旋
                continue

            parts = line.split()
            row = {
                "atom_1" : parts[1],
                "atom_2" : parts[2],
                "distance" : float(parts[3]),
                "icohp" : float(parts[7]),
                "spin" : spin_map[current_spin]
            }

            data_rows.append(row) # 这是一个字典，将上自旋和下自旋的数据都集合在一个字典中

        processed_data = {'atome_1' : data_rows[0]['atom_1'],
                          "atome_2" : data_rows[0]['atom_2'],
                          "distance" : data_rows[0]['distance'],
                          "icohp-up" : data_rows[0]['icohp'],
                          "icohp-down" : data_rows[1]['icohp']}

    return processed_data


In [39]:
parse_icohplist(filename='test.lobster')

{'atome_1': 'Mn37',
 'atome_2': 'O101',
 'distance': 1.94128,
 'icohp-up': -1.16752,
 'icohp-down': -1.4437}

In [ ]:
# 获取所有当前目录的
current_dir = os.getcwd()
subdirs = [d for d in os.listdir(current_dir) if os.path.isdir(os.path.join(current_dir, d))]

data = []
for subdir in subdirs:
    subdir_path = os.path.join(current_dir, subdir, "ICOHPCAR.lobster")
    data.append(parse_icohplist(subdir_path))
    cohp_filename = subdir + "_COHP.dat"
    old_cohp_path = os.path.join(current_dir, subdir, "COHPCAR.lobster")
    new_cohp_path = os.path.join(current_dir, cohp_filename)
    shutil.copy(old_cohp_path, new_cohp_path)

icohp_data = pd.DataFrame.from_records(data)
icohp_data.to_csv("all_data.csv", index=False)



